# Fed-SALGATE — Part 1 · Image Processing (FloodNet only)

**Federated Saliency-Consensus Gating for Drone-Based Flood Damage Assessment**

This is the **first part** of a 3-part pipeline. Part 1 does **image processing only**
(no model, no training):

1. Attach the **FloodNet** dataset exactly the way reference notebook
   `hybrid-cnn-vit-attention-u-net` does it:
   - **Train** ← `FloodNet Challenge - Track 1` → `Train/Labeled/{Flooded,Non-Flooded}`
   - **Val** ← `floodnet-dataset-for-segmentation-fixed-mask` → `data_finally/val`
   - **Val split** → real **val** + **test** (`train_test_split`, 70/30)
2. EDA: class mapping, per-class distribution, sample visualization, mask integrity.
3. Preprocessing: shape fix, **oversampling** of rare classes (hybrid approach).
4. Transforms + `Dataset` / `DataLoader` (optimized for GPU).
5. **Non-IID federated partitioning** (Dirichlet) → `client_indices.pkl`.
6. Save everything under a shared `fed_salgate_outputs/` folder that **Part 2 and
   Part 3** read back from any device.

**How to run on Kaggle (free T4):**
- Add the two datasets used below via *Add Data*.
- Run all cells in the background. Outputs are auto-saved under
  `/kaggle/working/fed_salgate_outputs` (also downloadable from the Output tab).
- Then attach **this notebook's Output** to Part 2 and Part 3 notebook runs.

Only **FloodNet** is used here (a single dataset domain), so client/val/test splits come
from the same UAV flood-imagery distribution — the fair setting for our FSCG study.

## ⚙️ Install packages

In [ ]:
import subprocess, sys

packages = [
    "albumentations==1.3.1",
    "scikit-learn",
    "opendatasets",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Packages ready.")

## 📚 Imports

In [ ]:
import os, sys, json, random, warnings, time, pickle
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
print("Imports OK. PyTorch", torch.__version__)

## ⚙️ Configuration (edit paths here if needed)

In [ ]:
# =========================================================
#  Global flags
# =========================================================
SMOKE_TEST = os.environ.get("FSCG_SMOKE") == "1"     # tiny mode for local testing
DEBUG_SHORT = os.environ.get("FSCG_DEBUG") == "1" or SMOKE_TEST

SEED        = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

IMG_SIZE    = int(os.environ.get("FSCG_IMG", "512"))
BATCH_SIZE  = int(os.environ.get("FSCG_BATCH", "4")) if not SMOKE_TEST else 2
NUM_CLIENTS = int(os.environ.get("FSCG_CLIENTS", "3"))
DIRICHLET_ALPHA = float(os.environ.get("FSCG_ALPHA", "0.5"))
VAL_SPLIT   = 0.3                     # val -> val + test split fraction
GT_VAL_FRAC = 0.30                    # fraction of val held out as REAL-GT final eval

IN_KAGGLE = os.path.isdir("/kaggle/input")

if IN_KAGGLE:
    WORKING_DIR = "/kaggle/working"
else:
    WORKING_DIR = os.path.join(os.path.dirname(os.getcwd()), "working")
    os.makedirs(WORKING_DIR, exist_ok=True)

OUTPUT_DIR = os.path.join(WORKING_DIR, "fed_salgate_outputs")
PLOT_DIR   = os.path.join(OUTPUT_DIR, "plots")
for d in [OUTPUT_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Mode      : {'KAGGLE' if IN_KAGGLE else 'LOCAL'}")
print(f"Smoke test: {SMOKE_TEST}")
print(f"IMG_SIZE  : {IMG_SIZE}   BATCH  : {BATCH_SIZE}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# =========================================================
#  Robust dataset discovery (works on Kaggle AND locally)
# =========================================================
def _find_root(search, keywords, require_files=None):
    """Recursively search `search` for a directory matching all keyword fragments."""
    hits = []
    require_files = require_files or []
    for root, dirs, files in os.walk(search):
        base = os.path.basename(root)
        if all(k.lower() in base.lower() for k in keywords):
            if all(os.path.exists(os.path.join(root, f)) for f in require_files):
                hits.append(root)
    return sorted(hits)

SEARCH_DIRS = ["/kaggle/input", os.path.expanduser("~")] if IN_KAGGLE else [os.path.expanduser("~")]

def find_floodnet_base():
    """Return the 'FloodNet Challenge - Track 1' root containing class_mapping.csv."""
    roots = ["/kaggle/input", "/kaggle/input/datasets"]
    for r in roots:
        if not os.path.isdir(r):
            continue
        for root, dirs, files in os.walk(r):
            dirname = os.path.basename(root)
            if "floodnet" in dirname.lower() and "challenge" in dirname.lower():
                for f in files:
                    if f == "class_mapping.csv":
                        # prefer the Track-1 root (has class_mapping + Train)
                        if os.path.isdir(os.path.join(root, "Train")):
                            return root
    return None

def find_secondary_base():
    """Return the 'floodnet-dataset-for-segmentation-fixed-mask' root."""
    for r in ["/kaggle/input", "/kaggle/input/datasets"]:
        if not os.path.isdir(r):
            continue
        for root, dirs, files in os.walk(r):
            if "floodnet" in os.path.basename(root).lower():
                if os.path.isdir(os.path.join(root, "data_finally")):
                    return root
            if os.path.basename(root) == "data_finally":
                return os.path.dirname(root)
    return None

FLOODNET_BASE = os.environ.get("FLOODNET_BASE", "") or find_floodnet_base()
SECONDARY_BASE = os.environ.get("SECONDARY_BASE", "") or find_secondary_base()

print(f"FLOODNET_BASE  : {FLOODNET_BASE}")
print(f"SECONDARY_BASE : {SECONDARY_BASE}")
if not FLOODNET_BASE:
    raise FileNotFoundError(
        "FloodNet Challenge - Track 1 not found. Add the Kaggle dataset "
        "'aletbm/aerial-imagery-dataset-floodnet-challenge' (or set FLOODNET_BASE).")
if not SECONDARY_BASE:
    raise FileNotFoundError(
        "floodnet-dataset-for-segmentation-fixed-mask not found. Add the Kaggle dataset "
        "'catashiro31/floodnet-dataset-for-segmentation-fixed-mask' (or set SECONDARY_BASE).")

## 🗂️ 1 · Class mapping (exactly as the reference notebook does)

In [ ]:
# Reference approach: class_mapping.csv -> columns 3:5 are (Class Index, Class Name)
df = pd.read_csv(os.path.join(FLOODNET_BASE, "class_mapping.csv"))

seg_part = df.iloc[:, 3:5]
seg_part.columns = ["Class Index", "Class Name"]
seg_part = seg_part.dropna(subset=["Class Index"])
seg_part = seg_part[seg_part["Class Index"].astype(str).str.isnumeric()]

class_mapping = seg_part.reset_index(drop=True)
class_mapping["Class Index"] = class_mapping["Class Index"].astype(int)

CLASS_NAMES = class_mapping["Class Name"].tolist()
NUM_CLASSES = len(class_mapping)

print(class_mapping.to_string(index=False))
print("Num classes:", NUM_CLASSES)

# Canonical FloodNet Track-1 semantic classes (used as fallback / checks):
_REF_NAMES = ["Background", "Building-Flooded", "Building-NonFlooded",
              "Road-Flooded", "Road-NonFlooded", "Water",
              "Tree", "Vehicle", "Pool", "Grass"]
if len(CLASS_NAMES) != 10:
    CLASS_NAMES = _REF_NAMES
    NUM_CLASSES = len(CLASS_NAMES)
    print("Fell back to canonical FloodNet 10-class list.")

assert NUM_CLASSES == 10, f"Expected 10 classes, got {NUM_CLASSES}"

COLOR_MAP = {
    0: [0, 0, 0], 1: [255, 0, 0], 2: [128, 0, 0],
    3: [255, 255, 0], 4: [128, 128, 0], 5: [0, 0, 255],
    6: [0, 128, 0], 7: [128, 128, 128], 8: [0, 255, 255], 9: [0, 255, 0],
}
print("Color map + class mapping ready.")

## 🚚 2 · Load TRAIN data — `Train/Labeled/{Flooded,Non-Flooded}` (reference style)

In [ ]:
def discover_masks(msk_dir, stem):
    """Find a mask file for image `stem` (reference naming: <stem>_lab.png)."""
    if not os.path.isdir(msk_dir):
        return None
    exts = (".png", ".jpg", ".jpeg", ".tif")
    # 1) primary reference naming
    for suf in ("_lab", "_mask", "_gt", "_label", ""):
        for e in exts:
            cand = os.path.join(msk_dir, f"{stem}{suf}{e}")
            if os.path.exists(cand):
                return cand
    # 2) fallback: any mask whose name subsumes the stem
    stem2 = os.path.splitext(os.path.basename(stem))[0]
    for f in os.listdir(msk_dir):
        if not f.lower().endswith(exts):
            continue
        fstem = os.path.splitext(f)[0]
        if fstem == stem2 or fstem.startswith(stem2) or stem2.startswith(fstem):
            return os.path.join(msk_dir, f)
    return None

train_imgs, train_masks = [], []
TRAIN_PATH = os.path.join(FLOODNET_BASE, "Train", "Labeled")
categories = ["Flooded", "Non-Flooded"]

for cat in categories:
    img_dir = os.path.join(TRAIN_PATH, cat, "image")
    msk_dir = os.path.join(TRAIN_PATH, cat, "mask")
    if not os.path.isdir(img_dir):
        print(f"  WARN missing: {img_dir}")
        continue
    for file in sorted(os.listdir(img_dir)):
        if not file.lower().endswith((".jpg", ".jpeg", ".png", ".tif")):
            continue
        img_path = os.path.join(img_dir, file)
        stem = os.path.splitext(file)[0]
        mask_path = discover_masks(msk_dir, stem)
        if mask_path is not None:
            train_imgs.append(img_path)
            train_masks.append(mask_path)

print("Train samples (real GT):", len(train_imgs))
sample_meta = [{"img": i, "mask": m} for i, m in zip(train_imgs, train_masks)]
with open(os.path.join(OUTPUT_DIR, "train_samples.json"), "w") as f:
    json.dump([{ "img": s["img"], "mask": s["mask"], "label": "flooded" if "Flooded" in s["img"] else "nonflooded"} for s in sample_meta], f, indent=2)
print("Saved train_samples.json")

## 🚚 3 · Load VAL + split into real `val` / `test` (reference style)

In [ ]:
VAL_IMG_DIR  = os.path.join(SECONDARY_BASE, "data_finally", "val", "image")
VAL_MASK_DIR = os.path.join(SECONDARY_BASE, "data_finally", "val", "mask")

print(f"VAL_IMG_DIR : {VAL_IMG_DIR}")
print(f"VAL_MASK_DIR: {VAL_MASK_DIR}")

val_image_paths, val_mask_paths = [], []
if os.path.isdir(VAL_MASK_DIR):
    mask_dict = {os.path.splitext(f)[0]: f for f in os.listdir(VAL_MASK_DIR)}
else:
    mask_dict = {}

for img_file in sorted(os.listdir(VAL_IMG_DIR)):
    if not img_file.lower().endswith((".jpg", ".jpeg", ".png", ".tif")):
        continue
    img_id = os.path.splitext(img_file)[0]
    if img_id in mask_dict:
        val_image_paths.append(os.path.join(VAL_IMG_DIR, img_file))
        val_mask_paths.append(os.path.join(VAL_MASK_DIR, mask_dict[img_id]))

print("Matched val samples (real GT):", len(val_image_paths))

# Split val -> val(70%) + test(30%) -- reference train_test_split
val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    val_image_paths, val_mask_paths,
    test_size=VAL_SPLIT, random_state=SEED, shuffle=True)

print("\nFinal split:")
print("  Train:", len(train_imgs))
print("  Val  :", len(val_imgs))
print("  Test :", len(test_imgs))

val_samples  = [{"img": i, "mask": m} for i, m in zip(val_imgs,  val_masks)]
test_samples = [{"img": i, "mask": m} for i, m in zip(test_imgs, test_masks)]
with open(os.path.join(OUTPUT_DIR, "val_samples.json"),  "w") as f:
    json.dump(val_samples, f, indent=2)
with open(os.path.join(OUTPUT_DIR, "test_samples.json"), "w") as f:
    json.dump(test_samples, f, indent=2)
print("Saved val_samples.json + test_samples.json")

## ✅ 4 · Dataset sanity checks (mask integrity + class coverage)

In [ ]:
def count_images_per_class(mask_paths):
    counts = Counter()
    for m in tqdm(mask_paths, desc="Counting classes", leave=False):
        mk = np.array(Image.open(m))
        if mk.ndim == 3:
            mk = mk[:, :, 0]
        for c in np.unique(mk):
            counts[int(c)] += 1
    return counts

def validate_masks(mask_paths, tag):
    bad = 0
    for m in mask_paths:
        mk = np.array(Image.open(m))
        if mk.ndim == 3:
            mk = mk[:, :, 0]
        if not set(np.unique(mk).tolist()).issubset(set(range(NUM_CLASSES))):
            bad += 1
    print(f"[{tag}] masks with out-of-range class ids: {bad}")
    assert bad == 0, f"Mask integrity failure in {tag}!"
    return True

validate_masks(train_masks, "train")
validate_masks(val_masks, "val")
validate_masks(test_masks, "test")

train_dist = count_images_per_class(train_masks)
print("\nPer-class #images appearing in train masks:")
for c in range(NUM_CLASSES):
    print(f"  {c:2d} {CLASS_NAMES[c]:<22} {train_dist.get(c, 0)}")

all_classes_present = all(train_dist.get(c, 0) > 0 for c in range(NUM_CLASSES))
print("\nAll 10 classes present in train:", all_classes_present)

## 📊 5 · EDA — visualize ground-truth samples

In [ ]:
def decode_mask_rgb(mask):
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls_id, color in COLOR_MAP.items():
        rgb[mask == cls_id] = color
    return rgb

def visualize_samples(img_paths, mask_paths, n=4, save_name=None):
    n = min(n, len(img_paths))
    fig, axes = plt.subplots(n, 3, figsize=(14, 4.2 * n))
    if n == 1:
        axes = axes[None, :]
    for i in range(n):
        img  = cv2.cvtColor(cv2.imread(img_paths[i]), cv2.COLOR_BGR2RGB)
        msk  = np.array(Image.open(mask_paths[i]))
        if msk.ndim == 3:
            msk = msk[:, :, 0]
        axes[i, 0].imshow(img);              axes[i, 0].set_title(f"Sample {i+1} - Image");     axes[i, 0].axis("off")
        axes[i, 1].imshow(msk, cmap="tab10"); axes[i, 1].set_title(f"Sample {i+1} - GT");      axes[i, 1].axis("off")
        axes[i, 2].imshow(cv2.addWeighted(img, 0.6, decode_mask_rgb(msk), 0.5, 0));
        axes[i, 2].set_title(f"Sample {i+1} - Overlay"); axes[i, 2].axis("off")
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(PLOT_DIR, save_name), dpi=120, bbox_inches="tight")
    plt.show()

visualize_samples(val_imgs, val_masks, n=2, save_name="01_samples_val.png")
visualize_samples(test_imgs, test_masks, n=2, save_name="02_samples_test.png")
print("EDA plots saved.")

## 📊 5b · Extra EDA — resolution, aspect ratio, channel statistics, legend

In [ ]:
import PIL.Image as _PILIm

def _shapes(paths, cap=400):
    ws, hs, ar = [], [], []
    for p in paths[:cap]:
        im = _PILIm.open(p)
        w, h = im.size
        ws.append(w); hs.append(h); ar.append(w / max(1, h))
    return ws, hs, ar

shapes = {}
for tag in ("train", "val", "test"):
    paths = {"train": train_imgs, "val": val_imgs, "test": test_imgs}[tag]
    shapes[tag] = _shapes(paths)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for j, (name, key) in enumerate([("Width (px)", "w"), ("Height (px)", "h"), ("Aspect ratio W/H", "ar")]):
    for t, s in shapes.items():
        ws, hs, ar = s
        data = ws if key == "w" else (hs if key == "h" else ar)
        axes[j].hist(data, bins=28, alpha=0.55, label=f"{t} (n={len(data)})")
    axes[j].set_title(f"Image {name} distribution")
    axes[j].legend(fontsize=8); axes[j].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01b_resolution_aspect.png"), dpi=130, bbox_inches="tight")
plt.show()
print("01b_resolution_aspect.png saved.")

# channel statistics -- Flooded vs Non-Flooded
def channel_stats(paths, n=40):
    rng = np.random.RandomState(SEED)
    n = min(n, len(paths))
    idx = np.sort(rng.choice(len(paths), size=n, replace=False))
    means = []
    for i in idx:
        img = cv2.imread(paths[i])
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
        means.append(img.reshape(-1, 3).mean(0))
    means = np.array(means)
    return means.mean(0), means.std(0)

flood = [p for p in train_imgs if "Flooded" in p]
nonf  = [p for p in train_imgs if "Non-Flooded" in p]
stats = {}
if flood:
    stats["Flooded"] = channel_stats(flood)
if nonf:
    stats["Non-Flooded"] = channel_stats(nonf)

fig, ax = plt.subplots(figsize=(9, 5))
xpos = np.arange(3)
ch_names = ["Red", "Green", "Blue"]
for k, (tag, (mu, sd)) in enumerate(stats.items()):
    offset = (k - 0.5) * 0.30
    ax.bar(xpos + offset, mu, width=0.28, yerr=sd, capsize=3, label=tag, alpha=0.85)
ax.set_xticks(xpos); ax.set_xticklabels(ch_names)
ax.set_ylabel("Mean pixel value [0-255]")
ax.set_title("Mean RGB intensity: Flooded vs Non-Flooded train images")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01c_channel_stats.png"), dpi=130, bbox_inches="tight")
plt.show()
print("01c_channel_stats.png saved.")

# class legend (matches COLOR_MAP everywhere)
import matplotlib.patches as mpatches
fig, ax = plt.subplots(figsize=(11, 2.6))
patches = [mpatches.Patch(color=np.array(COLOR_MAP[c]) / 255.0,
                          label=f"{c} {CLASS_NAMES[c]}") for c in range(NUM_CLASSES)]
ax.legend(handles=patches, loc="center", ncol=5, fontsize=9, frameon=False)
ax.axis("off")
plt.savefig(os.path.join(PLOT_DIR, "01d_class_legend.png"), dpi=130, bbox_inches="tight")
plt.show()
print("01d_class_legend.png saved.")

## ⚖️ 6 · Class-imbalance handling — oversampling (hybrid approach)

In [ ]:
def build_class_to_images(mask_paths):
    class_to_images = defaultdict(list)
    for idx, m in enumerate(tqdm(mask_paths, desc="Scanning masks")):
        mk = np.array(Image.open(m))
        if mk.ndim == 3:
            mk = mk[:, :, 0]
        for c in np.unique(mk):
            class_to_images[int(c)].append(idx)
    return class_to_images

def oversample_segmentation(imgs, masks, target_ratio=1.0, seed=SEED):
    """Oversample so every class appears in ~target_ratio * max images."""
    rng = np.random.RandomState(seed)
    class_to_images = build_class_to_images(masks)
    class_counts = {c: len(v) for c, v in class_to_images.items()}
    valid = [c for c in range(NUM_CLASSES) if class_counts.get(c, 0) > 0]
    if not valid:
        return list(imgs), list(masks), [False] * len(imgs)
    max_count  = max(class_counts[c] for c in valid)
    present    = len(valid)
    target_count = int(max_count * target_ratio * (present / NUM_CLASSES)) if present < NUM_CLASSES else int(max_count * target_ratio)
    target_count = max(target_count, max_count)

    out_imgs, out_masks, flags = [], [], []
    for i in range(len(imgs)):
        out_imgs.append(imgs[i]); out_masks.append(masks[i]); flags.append(False)

    for c in range(NUM_CLASSES):
        idxs = class_to_images.get(c, [])
        cur = len(idxs)
        if cur == 0 or cur >= target_count:
            continue
        need = target_count - cur
        extra = rng.choice(idxs, size=need, replace=True)
        for ei in extra:
            out_imgs.append(imgs[ei]); out_masks.append(masks[ei]); flags.append(True)

    print(f"\nBefore oversampling:")
    for c in range(NUM_CLASSES):
        print(f"  Class {c}: {class_counts.get(c, 0)}")
    print(f"\nTotal images after oversampling: {len(out_imgs)}")
    print(f"Total oversampled (flagged): {sum(flags)}")
    return out_imgs, out_masks, flags

oversampled_imgs, oversampled_masks, oversample_flags = oversample_segmentation(
    train_imgs, train_masks, target_ratio=1.0, seed=SEED)
print("Oversamping done. lengths:",
      len(oversampled_imgs), len(oversampled_masks), len(oversample_flags))

# --- persist oversampled train set + per-sample copy flags so Part 2 reuses
#     the same balanced dataset instead of re-deriving from originals only ---
oversampled_samples = [{"img": i, "mask": m}
                       for i, m in zip(oversampled_imgs, oversampled_masks)]
with open(os.path.join(OUTPUT_DIR, "train_samples_oversampled.json"), "w") as f:
    json.dump(oversampled_samples, f, indent=2)
with open(os.path.join(OUTPUT_DIR, "oversample_flags.json"), "w") as f:
    json.dump([bool(x) for x in oversample_flags], f, indent=2)
print(f"Saved train_samples_oversampled.json ({len(oversampled_samples)} samples) ")
print(f"      + oversample_flags.json ({sum(oversample_flags)} copies).")

## ⚖️ 6b · Class-imbalance table + oversampling before/after figures

In [ ]:
# pixel-level class distribution per split (proportion of labelled pixels)
def pixel_class_fraction(mask_paths):
    counts = np.zeros(NUM_CLASSES)
    for m in mask_paths:
        mk = np.array(Image.open(m))
        if mk.ndim == 3:
            mk = mk[:, :, 0]
        counts += np.bincount(mk.ravel(), minlength=NUM_CLASSES)[:NUM_CLASSES]
    return counts / (counts.sum() + 1e-8)

val_dist   = count_images_per_class(val_masks)
test_dist  = count_images_per_class(test_masks)

split_counts = pd.DataFrame({
    "Train (#images)":  [train_dist.get(c, 0) for c in range(NUM_CLASSES)],
    "Val (#images)":    [val_dist.get(c, 0)  for c in range(NUM_CLASSES)],
    "Test (#images)":   [test_dist.get(c, 0) for c in range(NUM_CLASSES)],
}, index=[f"{c} {CLASS_NAMES[c]}" for c in range(NUM_CLASSES)])
print(split_counts.to_string())

pix_split = pd.DataFrame({
    "Train": pixel_class_fraction(train_masks),
    "Val":   pixel_class_fraction(val_masks),
    "Test":  pixel_class_fraction(test_masks),
}, index=[f"{c} {CLASS_NAMES[c]}" for c in range(NUM_CLASSES)])
print("\nPixel-level class fractions:")
print(pix_split.round(4).to_string())

os.makedirs(os.path.join(OUTPUT_DIR, "tables"), exist_ok=True)
split_counts.to_csv(os.path.join(OUTPUT_DIR, "tables", "p1_split_image_counts.csv"))
pix_split.to_csv(os.path.join(OUTPUT_DIR, "tables", "p1_pixel_class_fractions.csv"))
print("Saved tables: p1_split_image_counts.csv, p1_pixel_class_fractions.csv")

# figure 04 -- image-presence counts per split (grouped bars)
fig, ax = plt.subplots(figsize=(12, 5))
xs = np.arange(NUM_CLASSES); w = 0.8 / 3
for j, key in enumerate(["Train (#images)", "Val (#images)", "Test (#images)"]):
    ax.bar(xs + (j - 1) * w, split_counts[key], width=w, label=key.split(" (")[0])
ax.set_xticks(xs); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_ylabel("#images containing class"); ax.set_title("Per-class image presence by split")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_split_image_counts.png"), dpi=130, bbox_inches="tight")
plt.show()
print("04_split_image_counts.png saved.")

# figure 05 -- oversampling before/after
after_dist = count_images_per_class(oversampled_masks)
fig, ax = plt.subplots(figsize=(12, 5))
before = np.array([train_dist.get(c, 0) for c in range(NUM_CLASSES)])
after  = np.array([after_dist.get(c, 0) for c in range(NUM_CLASSES)])
ax.bar(xs - 0.2, before, width=0.4, label="Before oversampling")
ax.bar(xs + 0.2, after,  width=0.4, label="After oversampling")
ax.set_xticks(xs); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_ylabel("#images containing class"); ax.set_title("Class-balancing via oversampling (hybrid approach)")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "05_oversample_before_after.png"), dpi=130, bbox_inches="tight")
plt.show()
print("05_oversample_before_after.png saved.")

# figure 06 -- stacked pixel-class fractions per split
fig, ax = plt.subplots(figsize=(12, 5.5))
bottom = np.zeros(3)
colors = [np.array(COLOR_MAP[c]) / 255.0 for c in range(NUM_CLASSES)]
for c in range(NUM_CLASSES):
    vals = [pix_split.iloc[c]["Train"], pix_split.iloc[c]["Val"], pix_split.iloc[c]["Test"]]
    ax.bar(["Train", "Val", "Test"], vals, bottom=bottom, width=0.5,
           label=CLASS_NAMES[c], color=colors[c])
    bottom += np.array(vals)
ax.set_ylabel("Pixel fraction"); ax.set_title("Class composition of FloodNet splits (pixel level)")
ax.legend(loc="upper right", fontsize=8, ncol=2); ax.grid(alpha=0.2, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "06_pixel_class_split.png"), dpi=130, bbox_inches="tight")
plt.show()
print("06_pixel_class_split.png saved.")

## 🎨 7 · Augmentation transforms (reference pipeline)

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

aug_normal = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.1, 0.1), rotate=(-30, 30), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.RandomGamma(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)

aug_strong = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(translate_percent=0.1, scale=(0.8, 1.2), rotate=(-45, 45), p=0.7),
    A.OneOf([A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.5),
             A.GridDistortion(p=0.5)], p=0.4),
    A.OneOf([A.RandomBrightnessContrast(p=0.5),
             A.HueSaturationValue(p=0.5),
             A.CLAHE(p=0.5)], p=0.5),
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)

print("Transforms defined. IMG_SIZE =", IMG_SIZE)

## 🧬 8 · Dataset classes + DataLoaders (optimized for GPU)

In [ ]:
class BalancedFloodDataset(Dataset):
    """Reference dataset. Original images use aug_normal, oversampled copies use aug_strong."""
    def __init__(self, image_paths, mask_paths, oversample_flags):
        self.image_paths    = image_paths
        self.mask_paths     = mask_paths
        self.flags          = oversample_flags

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            raise RuntimeError(f"Failed to read file idx={idx} "
                               f"img={self.image_paths[idx]} mask={self.mask_paths[idx]}")
        # FIX SHAPE (critical)
        if img.shape[:2] != mask.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        transform = aug_strong if self.flags[idx] else aug_normal
        augmented = transform(image=img, mask=mask)
        return augmented["image"], augmented["mask"].long()


class FloodDataset(Dataset):
    """Standard validation/test dataset (no augmentation)."""
    def __init__(self, image_paths, mask_paths, transform):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img  = cv2.imread(self.image_paths[idx])
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        augmented = self.transform(image=img, mask=mask)
        return augmented["image"], augmented["mask"].long()

In [ ]:
# cuDNN autotuner (Optimization 1)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    print("cudnn.benchmark = True")

train_dataset = BalancedFloodDataset(oversampled_imgs, oversampled_masks, oversample_flags)
val_dataset   = FloodDataset(val_imgs,  val_masks,  val_transform)
test_dataset  = FloodDataset(test_imgs, test_masks, val_transform)

print(f"train_dataset: {len(train_dataset)}")
print(f"val_dataset  : {len(val_dataset)}")
print(f"test_dataset : {len(test_dataset)}")

num_workers = 2 if not SMOKE_TEST else 0
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=num_workers, pin_memory=True,
                          persistent_workers=(num_workers > 0), drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=num_workers, pin_memory=True,
                          persistent_workers=(num_workers > 0))
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=num_workers, pin_memory=True,
                          persistent_workers=(num_workers > 0))

images, masks = next(iter(train_loader))
print("images shape:", images.shape, " dtype:", images.dtype)
print("masks  shape:", masks.shape,  " dtype:", masks.dtype)
assert images.shape[1] == 3 and masks.dtype == torch.int64
print("Loader shapes verified.")

## 🔀 9 · Non-IID federated partitioning (Dirichlet over pixel classes)

In [ ]:
class NonIIDPartitioner:
    """Dirichlet(alpha) per-class split into NUM_CLIENTS regional clients.
    alpha small -> extreme non-IID (each client sees few classes)."""
    def __init__(self, class_stat, num_clients, alpha=0.5, seed=SEED):
        self.class_stat = class_stat      # list/array of per-sample dominant class proxies
        self.num_clients = num_clients
        self.alpha = alpha
        self.seed = seed

    def run(self):
        rng = np.random.RandomState(self.seed)
        labels = np.asarray(self.class_stat).astype(int)
        client_indices = [[] for _ in range(self.num_clients)]
        for cls in range(NUM_CLASSES):
            idx_k = np.where(labels == cls)[0]
            if len(idx_k) == 0:
                continue
            rng.shuffle(idx_k)
            if self.num_clients == 1:
                client_indices[0].extend(idx_k.tolist())
                continue
            props = rng.dirichlet(np.repeat(self.alpha, self.num_clients))
            props = props / props.sum()
            splits = (np.cumsum(props) * len(idx_k)).astype(int)[:-1]
            for cid, chunk in enumerate(np.split(idx_k, splits)):
                client_indices[cid].extend(chunk.tolist())
        for ci in client_indices:
            rng.shuffle(ci)
        return client_indices


# Dominant class per real train image (used only for partitioning covariates)
def dominant_class(mask_path):
    mk = np.array(Image.open(mask_path))
    if mk.ndim == 3:
        mk = mk[:, :, 0]
    return int(np.bincount(mk.ravel(), minlength=NUM_CLASSES).argmax())

class_stat = [dominant_class(m) for m in tqdm(train_masks, desc="Dominant class")]

partitioner = NonIIDPartitioner(class_stat, NUM_CLIENTS, alpha=DIRICHLET_ALPHA, seed=SEED)
client_indices = partitioner.run()

print(f"\nDirichlet client sizes (alpha={DIRICHLET_ALPHA}):")
for ci, idx in enumerate(client_indices):
    print(f"  Client {ci}: {len(idx)} images")
assert sum(len(c) for c in client_indices) == len(train_imgs)
print("Partition size OK.")

In [ ]:
# Per-client pixel-class distributions (Gap-3 illustration)
def pixel_class_dist(mask_paths_subset, num_classes=NUM_CLASSES):
    counts = np.zeros(num_classes)
    for m in mask_paths_subset:
        mk = np.array(Image.open(m))
        if mk.ndim == 3:
            mk = mk[:, :, 0]
        counts += np.bincount(mk.ravel(), minlength=num_classes)[:num_classes]
    return counts / (counts.sum() + 1e-8)

fig, ax = plt.subplots(figsize=(12, 4.5))
xs = np.arange(NUM_CLASSES)
width = 0.8 / NUM_CLIENTS
for ci in range(NUM_CLIENTS):
    sub_imgs = [train_masks[i] for i in client_indices[ci]]
    dist = pixel_class_dist(sub_imgs)
    ax.bar(xs + ci * width, dist, width=width, label=f"Client {ci}")
ax.set_xticks(xs)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_ylabel("Pixel fraction")
ax.set_title(f"Non-IID per-client pixel-class distribution (alpha={DIRICHLET_ALPHA})")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_noniid_partition.png"), dpi=120, bbox_inches="tight")
plt.show()
print("03_noniid_partition.png saved.")

## 🔀 9b · Non-IID diagnostics — heatmap, client size, per-client entropy

In [ ]:
# heatmap: clients x classes (pixel fraction)
heat = np.zeros((NUM_CLIENTS, NUM_CLASSES))
for ci in range(NUM_CLIENTS):
    heat[ci] = pixel_class_fraction([train_masks[i] for i in client_indices[ci]])

fig, ax = plt.subplots(figsize=(11, 3 + 0.8 * NUM_CLIENTS))
im = ax.imshow(heat, aspect="auto", cmap="YlGnBu")
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(NUM_CLIENTS)); ax.set_yticklabels([f"Client {c}" for c in range(NUM_CLIENTS)])
for ci in range(NUM_CLIENTS):
    for cc in range(NUM_CLASSES):
        ax.text(cc, ci, f"{heat[ci, cc]:.2f}", ha="center", va="center", fontsize=7,
                color="black")
ax.set_title(f"Non-IID client class composition (pixel fraction) -- alpha={DIRICHLET_ALPHA}")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "07_noniid_heatmap.png"), dpi=130, bbox_inches="tight")
plt.show()
print("07_noniid_heatmap.png saved.")

# client sizes + Shannon entropy of class distribution
entropy = []
for ci in range(NUM_CLIENTS):
    e = -np.sum(np.where(heat[ci] > 0, heat[ci] * np.log2(heat[ci]), 0))
    entropy.append(e)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar([f"C{c}" for c in range(NUM_CLIENTS)], [len(c) for c in client_indices], color="tab:blue")
axes[0].set_ylabel("#samples"); axes[0].set_title("Client dataset sizes (non-IID)")
axes[0].grid(alpha=0.3, axis="y")
axes[1].bar([f"C{c}" for c in range(NUM_CLIENTS)], entropy, color="tab:orange")
axes[1].set_ylabel("Shannon entropy (bits)"); axes[1].set_title("Per-client class diversity")
axes[1].grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "08_noniid_clients.png"), dpi=130, bbox_inches="tight")
plt.show()
print("08_noniid_clients.png saved.")

client_summary = pd.DataFrame({
    "client": [f"C{c}" for c in range(NUM_CLIENTS)],
    "n_images": [len(c) for c in client_indices],
    "class_entropy_bits": [round(e, 3) for e in entropy],
})
client_summary.to_csv(os.path.join(OUTPUT_DIR, "tables", "p1_client_summary.csv"), index=False)
print("Saved tables/p1_client_summary.csv")

## 💾 10 · Save Part-1 outputs (read back by Part 2 & 3 from any device)

In [ ]:
# -- Held-out REAL-GT indices (a fixed slice of val, never used for training) --
n_gt = max(1, int(len(val_samples) * GT_VAL_FRAC))
gt_val_indices = list(range(0, n_gt))
print(f"Held-out GT eval set: {len(gt_val_indices)} val photos")

# -- class weights (inverse frequency, smoothed) for Part 2/3 losses --
tot = float(sum(train_dist.values()))
class_weights = [tot / max(1, train_dist.get(c, 1)) for c in range(NUM_CLASSES)]
class_weights = [w / max(class_weights) for w in class_weights]          # normalize to [0..1]
print("Class weights (normalized inverse frequency):")
print([round(w, 3) for w in class_weights])

config = {
    "SEED": SEED,
    "IMG_SIZE": IMG_SIZE,
    "BATCH_SIZE": BATCH_SIZE,
    "NUM_CLASSES": NUM_CLASSES,
    "CLASS_NAMES": CLASS_NAMES,
    "COLOR_MAP": {str(k): v for k, v in COLOR_MAP.items()},
    "NUM_CLIENTS": NUM_CLIENTS,
    "DIRICHLET_ALPHA": DIRICHLET_ALPHA,
    "VAL_SPLIT": VAL_SPLIT,
    "GT_VAL_FRAC": GT_VAL_FRAC,
    "FLOODNET_BASE": FLOODNET_BASE,
    "SECONDARY_BASE": SECONDARY_BASE,
    "train_count": len(train_imgs),
    "val_count": len(val_imgs),
    "test_count": len(test_imgs),
    "oversampled_train_count": len(oversampled_imgs),
    "class_weights": class_weights,
    "train_dist": {str(k): v for k, v in train_dist.items()},
    "REFERENCE": "hybrid-cnn-vit-attention-u-net",
}

with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "client_indices.pkl"), "wb") as f:
    pickle.dump(client_indices, f)

with open(os.path.join(OUTPUT_DIR, "gt_val_indices.pkl"), "wb") as f:
    pickle.dump(gt_val_indices, f)

# summary text for humans
summary_lines = [
    "Fed-SALGATE Part 1 - Image Processing Summary",
    "=" * 50,
    f"FloodNet base    : {FLOODNET_BASE}",
    f"Secondary base   : {SECONDARY_BASE}",
    f"Split            : train={config['train_count']}  val={config['val_count']}  test={config['test_count']}",
    f"Oversampled train: {config['oversampled_train_count']}",
    f"Clients (non-IID): {NUM_CLIENTS}  alpha={DIRICHLET_ALPHA}",
    f"Held-out GT      : {len(gt_val_indices)}",
    f"Img size         : {IMG_SIZE}",
]
with open(os.path.join(OUTPUT_DIR, "PART1_SUMMARY.txt"), "w") as f:
    f.write("\n".join(summary_lines))
print("\n".join(summary_lines))

## 🛡️ 10a · Leak audit — split disjointness & oversampling isolation

In [ ]:
import hashlib, datetime, platform

leak_issues = []

# 1) path-level disjointness between train / val / test
s_train, s_val, s_test = set(train_imgs), set(val_imgs), set(test_imgs)
for tag, A, B in [("train", s_train, s_val), ("train", s_train, s_test),
                  ("val", s_val, s_test)]:
    inter = A & B
    if inter:
        leak_issues.append(f"{tag} shares {len(inter)} sample paths with the other split")

# 2) oversampled copies are only duplicates of TRAIN paths (never injected into val/test)
ov_originals = [p for p, f in zip(oversampled_imgs, oversample_flags) if not f]
ov_copies    = [p for p, f in zip(oversampled_imgs, oversample_flags) if f]
if set(ov_originals) != s_train:
    leak_issues.append("oversample originals do not exactly match the train split")
if not (set(ov_copies) <= s_train):
    leak_issues.append("oversampled copies reference paths outside train")
if set(ov_copies) & (s_val | s_test):
    leak_issues.append("oversampled copies collided with val/test paths")

# 3) img/mask pairing sanity (every sample has its own mask, nothing shared)
for img, msk in zip(train_imgs + val_imgs + test_imgs,
                    train_masks + val_masks + test_masks):
    if not os.path.isfile(msk):
        leak_issues.append(f"missing mask {msk}")

leak_report = [
    f"train paths          : {len(s_train)}",
    f"val paths            : {len(s_val)}",
    f"test paths           : {len(s_test)}",
    f"train∩val            : {len(s_train & s_val)}",
    f"train∩test           : {len(s_train & s_test)}",
    f"val∩test             : {len(s_val & s_test)}",
    f"oversample originals : {len(ov_originals)} (must equal train)",
    f"oversample copies    : {len(ov_copies)} (train-only duplicates)",
]
print("\n".join(leak_report))

if leak_issues:
    for x in leak_issues:
        print("!! LEAK:", x)
    raise AssertionError(f"Masking/leak audit FAILED: {leak_issues}")

with open(os.path.join(OUTPUT_DIR, "tables", "p1_leak_audit.txt"), "w") as f:
    f.write("\n".join(leak_report) + "\nAUDIT PASSED\n")
print("Leak audit PASSED: train/val/test are disjoint, oversampling is train-only.")

## 📊 10b · Hyperparameter & reproducibility tracker (Part 1)

In [ ]:
hp = {
    "part": 1,
    "reference": config.get("REFERENCE", "hybrid-cnn-vit-attention-u-net"),
    "task": "FloodNet segmentation (10 classes) - non-IID prep",
    "seed": SEED,
    "img_size": IMG_SIZE,
    "num_classes": NUM_CLASSES,
    "num_clients": NUM_CLIENTS,
    "dirichlet_alpha": DIRICHLET_ALPHA,
    "val_split": VAL_SPLIT,
    "gt_val_frac": GT_VAL_FRAC,
    "train_count": len(train_imgs),
    "val_count": len(val_imgs),
    "test_count": len(test_imgs),
    "oversampled_train_count": len(oversampled_imgs),
    "oversampled_copies": int(sum(oversample_flags)),
    "oversample_target_ratio": 1.0,
    "aug_scheme": "Part-2 applies normal aug to originals, strong aug to copies",
    "image_norm": "ImageNet mean=(0.485,0.456,0.406) std=(0.229,0.224,0.225)",
    "label_smoothing_none": True,
    "python": platform.python_version(),
    "date": datetime.date.today().isoformat(),
}
hp_flat = pd.DataFrame({"hyperparameter": list(hp), "value": [str(v) for v in hp.values()]})
with open(os.path.join(OUTPUT_DIR, "tables", "p1_hyperparams.json"), "w") as f:
    json.dump(hp, f, indent=2, default=str)
hp_flat.to_csv(os.path.join(OUTPUT_DIR, "tables", "p1_hyperparams.csv"), index=False)
print("Part-1 hyperparameters (saved to tables/p1_hyperparams.{json,csv}):")
for k, v in hp.items():
    print(f"  {k:<28} {str(v)[:90]}")
print("\nAll Part-1 outputs saved under:", OUTPUT_DIR)

## 📦 10b · Export summary tables + ZIP the outputs

In [ ]:
# split summary table
split_summary = pd.DataFrame([{
    "split": "train", "n_images": len(train_imgs),
    "n_masks": len(train_masks), "n_gt_y": sum(train_dist.values()) or 0,
}, {
    "split": "val", "n_images": len(val_imgs), "n_masks": len(val_masks),
    "n_gt_y": sum(val_dist.values()) or 0,
}, {
    "split": "test", "n_images": len(test_imgs), "n_masks": len(test_masks),
    "n_gt_y": sum(test_dist.values()) or 0,
}])

split_summary.to_csv(os.path.join(OUTPUT_DIR, "tables", "p1_split_summary.csv"), index=False)
pd.DataFrame([{"config_key": k, "value": v} for k, v in config.items()]
             ).to_csv(os.path.join(OUTPUT_DIR, "tables", "p1_config.csv"), index=False)
print("Saved tables: p1_split_summary.csv, p1_config.csv")

# ZIP the (small) outputs so they can be transferred from any device
import zipfile
_zip_path = os.path.join(WORKING_DIR, "part1_outputs.zip")
with zipfile.ZipFile(_zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUTPUT_DIR):
        for f in files:
            if f.endswith(".pth"):
                continue
            p = os.path.join(root, f)
            z.write(p, os.path.relpath(p, WORKING_DIR))
size_mb = os.path.getsize(_zip_path) / 1e6
print(f"Zipped Part-1 outputs -> {_zip_path}  ({size_mb:.2f} MB)")

In [ ]:
print("=" * 60)
print("PART 1 OUTPUT INVENTORY")
print("=" * 60)
OUT_FILES = {
    "config.json": 0, "train_samples.json": 0, "val_samples.json": 0,
    "test_samples.json": 0, "client_indices.pkl": 0, "gt_val_indices.pkl": 0,
}
for name in OUT_FILES:
    p = os.path.join(OUTPUT_DIR, name)
    if os.path.exists(p):
        OUT_FILES[name] = os.path.getsize(p)
for k, v in OUT_FILES.items():
    print(f"  {k:<22} {v/1024:8.1f} KB")

print("\nPlots:", sorted(os.listdir(PLOT_DIR)))
print("\nDONE -- Part 1 finished. Attach this notebook's Output to Part 2.")